In [1]:
# ============================================================
# 16_AURORA_mechanism_control_experiments.ipynb
# Corrected single-cell standalone version
#
# Fixes:
# - Bootstrap samples can have integer indexes.
# - performance_metrics_from_returns() safely handles datetime and non-datetime indexes.
# - Keeps the mechanism-control design:
#   1. Static 44.35% equity / 55.65% cash baseline
#   2. Constant-average uncertainty AURORA
#   3. 20d-only, 60d-only, equal-blend AURORA
#   4. Shuffled-probability AURORA controls
#
# Outputs:
# - table_S27_mechanism_control_performance.csv
# - table_S28_mechanism_control_bootstrap.csv
# - table_S29_probability_control_diagnostics.csv
# - table_S30_exposure_control_diagnostics.csv
#
# Research backtest only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import copy
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import minimize
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
    print("scipy.optimize not available. Optimizer will use fallback allocations.")

# ============================================================
# 1. Paths and global settings
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

NOTEBOOK07_RUN_ID = "20260624_031817"
NOTEBOOK07_ROOT = OUTPUT_ROOT / "purged_walk_forward_models" / f"run_{NOTEBOOK07_RUN_ID}"
NOTEBOOK08_INPUT_INDEX = NOTEBOOK07_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "aurora_mechanism_controls" / f"run_{RUN_ID}"

RETURN_DIR = RUN_ROOT / "returns"
WEIGHT_DIR = RUN_ROOT / "weights"
TABLE_RUN_DIR = RUN_ROOT / "tables"
DIAG_DIR = RUN_ROOT / "diagnostics"
REPORT_RUN_DIR = RUN_ROOT / "reports"
PLOT_DIR = RUN_ROOT / "plots"

for d in [
    RUN_ROOT,
    RETURN_DIR,
    WEIGHT_DIR,
    TABLE_RUN_DIR,
    DIAG_DIR,
    REPORT_RUN_DIR,
    PLOT_DIR,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "CASH"
ALL_ASSETS = ETF_UNIVERSE + [CASH_COL]

CLASS_LABELS = [0, 1, 2, 3, 4]

ANNUALIZATION_DAYS = 252
INITIAL_CAPITAL = 1.0
TRANSACTION_COST_RATE = 0.0010
REBALANCE_FREQUENCY = "monthly"

STRICT_START = "2024-11-27"
STRICT_END = "2026-03-25"

BOOTSTRAP_REPLICATIONS = 5000
BOOTSTRAP_BLOCK_LENGTH = 20
BOOTSTRAP_RANDOM_SEED = 20260714

SHUFFLE_SEEDS = list(range(1000, 1010))

AURORA_REPORTED_AVG_CASH = 0.5565
AURORA_REPORTED_AVG_EQUITY = 1.0 - AURORA_REPORTED_AVG_CASH

STATIC_EXPOSURE_MATCHED_WEIGHTS = {
    "0050": AURORA_REPORTED_AVG_EQUITY / 4.0,
    "006208": AURORA_REPORTED_AVG_EQUITY / 4.0,
    "00692": AURORA_REPORTED_AVG_EQUITY / 4.0,
    "00881": AURORA_REPORTED_AVG_EQUITY / 4.0,
    "CASH": AURORA_REPORTED_AVG_CASH,
}

FALLBACK_CONSTRAINED_EQUAL = {
    "0050": 0.30,
    "006208": 0.30,
    "00692": 0.25,
    "00881": 0.15,
    "CASH": 0.00,
}

FALLBACK_DEFENSIVE = {
    "0050": 0.24,
    "006208": 0.24,
    "00692": 0.20,
    "00881": 0.12,
    "CASH": 0.20,
}

BASE_UAMV_B_CONFIG = {
    "strategy_name": "UAMV_B_more60_defensive",
    "alpha_20d": 0.30,
    "alpha_60d": 0.70,
    "lookback_mu": 63,
    "lookback_cov": 126,
    "mean_shrinkage_to_zero": 0.60,
    "momentum_weight": 0.40,
    "base_risk_aversion": 10.0,
    "uncertainty_risk_multiplier": 2.5,
    "bearish_risk_multiplier": 2.0,
    "turnover_penalty": 0.25,
    "regime_tilt_strength": 0.25,
    "max_etf_weight": 0.45,
    "max_00881_weight": 0.30,
    "max_cash_weight": 0.60,
    "min_cash_weight": 0.00,
    "risk_aversion_mode": "dynamic",
    "constant_average_U": None,
    "constant_average_B": None,
}

print("=" * 80)
print("AURORA-TWETF Notebook 16: Mechanism-control experiments")
print("=" * 80)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("Notebook 07 input index:", NOTEBOOK08_INPUT_INDEX)
print("=" * 80)

if not NOTEBOOK08_INPUT_INDEX.exists():
    raise FileNotFoundError(f"Missing probability input index: {NOTEBOOK08_INPUT_INDEX}")

# ============================================================
# 2. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def clean_symbol_name(x):
    x = str(x)
    x = x.replace(".TW", "")
    x = x.replace(".TWO", "")
    x = x.replace("TW_", "")
    return x

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("+", "plus")
    )

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        df.index = pd.to_datetime(df.index)

    df.index.name = "date"
    return df.sort_index()

def load_etf_return_panel():
    candidates = [
        PANEL_DIR / "AURORA_etf_return_panel.parquet",
        PANEL_DIR / "AURORA_etf_returns_panel.parquet",
        PANEL_DIR / "AURORA_return_panel.parquet",
        MODELING_DIR / "AURORA_etf_return_panel.parquet",
    ]

    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            "Could not find ETF return panel. Tried:\n"
            + "\n".join(str(p) for p in candidates)
        )

    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df.sort_index()
    df = df.rename(columns={c: clean_symbol_name(c) for c in df.columns})

    missing = [s for s in ETF_UNIVERSE if s not in df.columns]
    if missing:
        raise ValueError(
            f"ETF return panel found at {path}, but missing ETF columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    df = df[ETF_UNIVERSE].copy()
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df, path

def proba_cols():
    return [f"proba_class_{i}" for i in CLASS_LABELS]

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def normalize_vector(w):
    w = pd.Series(w, index=ALL_ASSETS, dtype=float)
    w = w.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    w[w < 0] = 0.0

    if w.sum() <= 0:
        w = pd.Series(FALLBACK_CONSTRAINED_EQUAL, dtype=float).reindex(ALL_ASSETS).fillna(0.0)

    return w / w.sum()

def normalize_rows(df):
    out = df.copy()
    out = out.reindex(columns=ALL_ASSETS).fillna(0.0)
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out[out < 0] = 0.0

    row_sums = out.sum(axis=1)
    zero_mask = row_sums <= 0

    if zero_mask.any():
        out.loc[zero_mask, ETF_UNIVERSE] = 1.0 / len(ETF_UNIVERSE)
        out.loc[zero_mask, CASH_COL] = 0.0
        row_sums = out.sum(axis=1)

    out = out.div(row_sums, axis=0)
    return out

def latest_fold_deduplicate(proba_df, split_filter=None):
    df = proba_df.copy()

    if split_filter is not None and "split" in df.columns:
        if isinstance(split_filter, str):
            split_filter = [split_filter]
        df = df[df["split"].isin(split_filter)].copy()

    if df.empty:
        return df

    df = df.reset_index()

    if "fold_id" not in df.columns:
        df["fold_id"] = "WF0"

    df["fold_number"] = (
        df["fold_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .fillna("0")
        .astype(int)
    )

    split_priority = {"train": 0, "validation": 1, "test": 2}
    if "split" in df.columns:
        df["split_priority"] = df["split"].map(split_priority).fillna(0).astype(int)
    else:
        df["split_priority"] = 0

    df = df.sort_values(["date", "split_priority", "fold_number"])
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.set_index("date").sort_index()
    df.index.name = "date"

    return df.drop(columns=["fold_number", "split_priority"], errors="ignore")

# ============================================================
# 3. Probability features
# ============================================================

def probability_features_from_blend(p20, p60, config):
    common = p20.index.intersection(p60.index).sort_values()

    p20 = p20.loc[common].copy()
    p60 = p60.loc[common].copy()

    prob20 = normalize_proba(p20[proba_cols()].values)
    prob60 = normalize_proba(p60[proba_cols()].values)

    alpha20 = float(config["alpha_20d"])
    alpha60 = float(config["alpha_60d"])

    if alpha20 + alpha60 <= 0:
        alpha20, alpha60 = 0.5, 0.5
    else:
        s = alpha20 + alpha60
        alpha20, alpha60 = alpha20 / s, alpha60 / s

    prob = alpha20 * prob20 + alpha60 * prob60
    prob = normalize_proba(prob)

    class_values = np.asarray(CLASS_LABELS, dtype=float)
    expected_class = prob @ class_values

    clipped = np.clip(prob, 1e-12, 1.0)
    entropy = -np.sum(clipped * np.log(clipped), axis=1)
    normalized_entropy = entropy / np.log(len(CLASS_LABELS))
    confidence = 1.0 - normalized_entropy

    sorted_p = np.sort(prob, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    ordinal_variance = (prob @ (class_values ** 2)) - expected_class ** 2

    out = pd.DataFrame(index=common)
    out.index.name = "date"
    out["expected_class"] = expected_class
    out["entropy"] = entropy
    out["normalized_entropy"] = normalized_entropy
    out["confidence_score"] = confidence
    out["probability_margin"] = margin
    out["ordinal_variance"] = ordinal_variance
    out["p_strong_bear"] = prob[:, 0]
    out["p_bear"] = prob[:, 1]
    out["p_neutral"] = prob[:, 2]
    out["p_bull"] = prob[:, 3]
    out["p_strong_bull"] = prob[:, 4]
    out["p_bearish"] = prob[:, 0] + prob[:, 1]
    out["p_bullish"] = prob[:, 3] + prob[:, 4]
    out["risk_on_score"] = (expected_class - 2.0) / 2.0

    for i, label in enumerate(CLASS_LABELS):
        out[f"proba_class_{label}"] = prob[:, i]

    return out

def shuffled_probability_inputs(p20, p60, evaluation_dates, seed):
    rng = np.random.default_rng(seed)

    common = (
        pd.DatetimeIndex(evaluation_dates)
        .intersection(p20.index)
        .intersection(p60.index)
        .sort_values()
    )

    perm = rng.permutation(len(common))
    source_dates = common[perm]

    p20_out = p20.copy()
    p60_out = p60.copy()

    p20_out.loc[common, proba_cols()] = p20.loc[source_dates, proba_cols()].values
    p60_out.loc[common, proba_cols()] = p60.loc[source_dates, proba_cols()].values

    return p20_out, p60_out

# ============================================================
# 4. AURORA optimizer
# ============================================================

def apply_caps_to_vector(w, config):
    w = normalize_vector(w)

    max_etf_weight = float(config["max_etf_weight"])
    max_00881_weight = float(config["max_00881_weight"])
    max_cash_weight = float(config["max_cash_weight"])
    min_cash_weight = float(config["min_cash_weight"])

    w[CASH_COL] = min(max(w[CASH_COL], min_cash_weight), max_cash_weight)

    for etf in ETF_UNIVERSE:
        cap = max_etf_weight
        if etf == "00881":
            cap = min(cap, max_00881_weight)
        w[etf] = min(w[etf], cap)

    if w.sum() <= 0:
        w = pd.Series(FALLBACK_CONSTRAINED_EQUAL, dtype=float).reindex(ALL_ASSETS).fillna(0.0)

    w = w / w.sum()

    for _ in range(10):
        excess = 0.0
        capped = []

        for etf in ETF_UNIVERSE:
            cap = max_etf_weight
            if etf == "00881":
                cap = min(cap, max_00881_weight)

            if w[etf] > cap:
                excess += w[etf] - cap
                w[etf] = cap
                capped.append(etf)

        if w[CASH_COL] > max_cash_weight:
            excess += w[CASH_COL] - max_cash_weight
            w[CASH_COL] = max_cash_weight
            capped.append(CASH_COL)

        if w[CASH_COL] < min_cash_weight:
            needed = min_cash_weight - w[CASH_COL]
            w[CASH_COL] = min_cash_weight
            risky_sum = w[ETF_UNIVERSE].sum()
            if risky_sum > 0:
                w[ETF_UNIVERSE] *= max(0.0, risky_sum - needed) / risky_sum

        if excess <= 1e-12:
            break

        eligible = []
        for asset in ALL_ASSETS:
            if asset in capped:
                continue
            if asset == CASH_COL:
                if w[asset] < max_cash_weight:
                    eligible.append(asset)
            elif asset == "00881":
                if w[asset] < min(max_etf_weight, max_00881_weight):
                    eligible.append(asset)
            else:
                if w[asset] < max_etf_weight:
                    eligible.append(asset)

        if not eligible:
            break

        eligible_sum = w[eligible].sum()
        if eligible_sum <= 0:
            for asset in eligible:
                w[asset] += excess / len(eligible)
        else:
            w[eligible] += excess * (w[eligible] / eligible_sum)

        w = w.clip(lower=0.0)
        w = w / w.sum()

    return normalize_vector(w)

def fallback_weight(features_row, config):
    p_bearish = float(features_row.get("p_bearish", 0.0))
    confidence = float(features_row.get("confidence_score", 0.5))

    if p_bearish > 0.45 and confidence > 0.25:
        base = pd.Series(FALLBACK_DEFENSIVE, dtype=float)
    else:
        base = pd.Series(FALLBACK_CONSTRAINED_EQUAL, dtype=float)

    return apply_caps_to_vector(base.reindex(ALL_ASSETS).fillna(0.0), config)

def estimate_moments_for_date(etf_returns, date, features_row, config):
    lookback_mu = int(config["lookback_mu"])
    lookback_cov = int(config["lookback_cov"])

    hist_all = etf_returns.loc[etf_returns.index < date, ETF_UNIVERSE].copy()

    if len(hist_all) < max(30, min(lookback_mu, lookback_cov) // 2):
        return None, None

    hist_mu = hist_all.tail(lookback_mu)
    hist_cov = hist_all.tail(lookback_cov)

    if len(hist_mu) < 20 or len(hist_cov) < 30:
        return None, None

    mean_short = hist_mu.mean().values
    momentum_return = (1.0 + hist_mu).prod().values - 1.0
    momentum_daily = momentum_return / max(len(hist_mu), 1)

    momentum_weight = float(config["momentum_weight"])
    shrink = float(config["mean_shrinkage_to_zero"])

    mu_risky = (
        (1.0 - momentum_weight) * mean_short
        + momentum_weight * momentum_daily
    )

    mu_risky = (1.0 - shrink) * mu_risky

    risk_on = float(features_row.get("risk_on_score", 0.0))
    p_bearish = float(features_row.get("p_bearish", 0.0))
    p_bullish = float(features_row.get("p_bullish", 0.0))

    tilt_strength = float(config["regime_tilt_strength"])

    tilt = pd.Series(0.0, index=ETF_UNIVERSE)
    tilt["0050"] += 0.15 * risk_on
    tilt["006208"] += 0.15 * risk_on
    tilt["00692"] += 0.05 * risk_on
    tilt["00881"] += 0.35 * risk_on + 0.15 * p_bullish - 0.20 * p_bearish
    tilt["00692"] += 0.10 * p_bearish

    realized_vol = hist_cov.std().replace(0.0, np.nan)
    vol_scale = realized_vol.median()

    if not np.isfinite(vol_scale) or vol_scale <= 0:
        vol_scale = 0.01

    mu_risky = mu_risky + tilt_strength * tilt.values * vol_scale / ANNUALIZATION_DAYS

    cov_risky = hist_cov.cov().values
    cov_risky = np.nan_to_num(cov_risky, nan=0.0, posinf=0.0, neginf=0.0)

    avg_var = np.mean(np.diag(cov_risky))
    if not np.isfinite(avg_var) or avg_var <= 0:
        avg_var = 1e-4

    cov_risky = cov_risky + np.eye(len(ETF_UNIVERSE)) * avg_var * 0.10

    mu = np.zeros(len(ALL_ASSETS), dtype=float)
    mu[:len(ETF_UNIVERSE)] = mu_risky
    mu[-1] = 0.0

    cov = np.zeros((len(ALL_ASSETS), len(ALL_ASSETS)), dtype=float)
    cov[:len(ETF_UNIVERSE), :len(ETF_UNIVERSE)] = cov_risky
    cov[-1, -1] = 1e-10

    return mu, cov

def risk_aversion_for_date(features_row, config):
    base = float(config["base_risk_aversion"])
    uncertainty_mult = float(config["uncertainty_risk_multiplier"])
    bearish_mult = float(config["bearish_risk_multiplier"])

    if config.get("risk_aversion_mode", "dynamic") == "constant_average":
        uncertainty = float(config["constant_average_U"])
        p_bearish = float(config["constant_average_B"])
    else:
        uncertainty = float(features_row.get("normalized_entropy", 0.5))
        p_bearish = float(features_row.get("p_bearish", 0.0))

    lam = base * (1.0 + uncertainty_mult * uncertainty + bearish_mult * p_bearish)
    return float(max(lam, 1e-6))

def optimize_single_date(mu, cov, prev_w, features_row, config):
    if mu is None or cov is None or not HAS_SCIPY:
        return fallback_weight(features_row, config), "fallback_no_moments_or_scipy"

    max_etf_weight = float(config["max_etf_weight"])
    max_00881_weight = float(config["max_00881_weight"])
    max_cash_weight = float(config["max_cash_weight"])
    min_cash_weight = float(config["min_cash_weight"])
    turnover_penalty = float(config["turnover_penalty"])

    lam = risk_aversion_for_date(features_row, config)
    prev = normalize_vector(prev_w).values

    bounds = []
    for asset in ALL_ASSETS:
        if asset == CASH_COL:
            bounds.append((min_cash_weight, max_cash_weight))
        elif asset == "00881":
            bounds.append((0.0, min(max_etf_weight, max_00881_weight)))
        else:
            bounds.append((0.0, max_etf_weight))

    def objective(w):
        w = np.asarray(w, dtype=float)
        expected_return = float(mu @ w)
        variance = float(w.T @ cov @ w)
        turnover_term = float(np.sum((w - prev) ** 2))
        utility = expected_return - lam * variance - turnover_penalty * turnover_term
        return -utility

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    x0 = apply_caps_to_vector(prev, config).values

    try:
        res = minimize(
            objective,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 300, "ftol": 1e-10, "disp": False},
        )

        if res.success and np.all(np.isfinite(res.x)):
            w = apply_caps_to_vector(res.x, config)
            return w, "optimized"

        w = fallback_weight(features_row, config)
        return w, f"fallback_optimizer_failed_{str(res.message)[:80]}"

    except Exception as e:
        w = fallback_weight(features_row, config)
        return w, f"fallback_exception_{repr(e)[:80]}"

def build_uamv_signal_weights(p20, p60, etf_returns, config, evaluation_dates):
    features = probability_features_from_blend(p20, p60, config)

    evaluation_dates = (
        pd.DatetimeIndex(evaluation_dates)
        .intersection(features.index)
        .intersection(etf_returns.index)
        .sort_values()
    )

    if len(evaluation_dates) == 0:
        raise ValueError(f"No evaluation dates for {config['strategy_name']}")

    rows = []
    diagnostics = []

    prev_w = pd.Series(FALLBACK_CONSTRAINED_EQUAL, dtype=float).reindex(ALL_ASSETS).fillna(0.0)
    prev_w = apply_caps_to_vector(prev_w, config)

    for dt in evaluation_dates:
        features_row = features.loc[dt]

        mu, cov = estimate_moments_for_date(
            etf_returns=etf_returns,
            date=dt,
            features_row=features_row,
            config=config,
        )

        w, status = optimize_single_date(
            mu=mu,
            cov=cov,
            prev_w=prev_w,
            features_row=features_row,
            config=config,
        )

        rows.append(w.values)

        diagnostics.append({
            "date": dt,
            "strategy_name": config["strategy_name"],
            "optimization_status": status,
            "risk_aversion": risk_aversion_for_date(features_row, config),
            "expected_class": float(features_row["expected_class"]),
            "p_bearish": float(features_row["p_bearish"]),
            "p_bullish": float(features_row["p_bullish"]),
            "normalized_entropy": float(features_row["normalized_entropy"]),
            "confidence_score": float(features_row["confidence_score"]),
            "ordinal_variance": float(features_row["ordinal_variance"]),
            "risk_aversion_mode": config.get("risk_aversion_mode", "dynamic"),
        })

        prev_w = w

    weight_df = pd.DataFrame(rows, index=evaluation_dates, columns=ALL_ASSETS)
    weight_df.index.name = "date"
    weight_df = normalize_rows(weight_df)

    diagnostic_df = pd.DataFrame(diagnostics).set_index("date").sort_index()
    diagnostic_df.index.name = "date"

    return weight_df, diagnostic_df, features.loc[evaluation_dates].copy()

# ============================================================
# 5. Backtest and metrics
# ============================================================

def get_rebalance_dates(index, frequency):
    idx = pd.DatetimeIndex(index).sort_values()

    if frequency == "monthly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.month])
    elif frequency == "quarterly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.quarter])
    elif frequency == "weekly":
        iso = idx.isocalendar()
        groups = pd.Series(idx, index=idx).groupby([iso.year, iso.week])
    elif frequency == "daily":
        return idx
    else:
        raise ValueError(f"Unsupported rebalance frequency: {frequency}")

    return pd.DatetimeIndex([values.iloc[0] for _, values in groups])

def expand_rebalance_weights_to_daily(signal_weight_df, daily_index, rebalance_dates):
    signal = signal_weight_df.reindex(columns=ALL_ASSETS).fillna(0.0).copy()
    signal = normalize_rows(signal)
    signal_idx = pd.DatetimeIndex(signal.index).sort_values()

    daily = pd.DataFrame(index=daily_index, columns=ALL_ASSETS, dtype=float)

    for i, reb_date in enumerate(rebalance_dates):
        if i + 1 < len(rebalance_dates):
            period_idx = daily_index[(daily_index >= reb_date) & (daily_index < rebalance_dates[i + 1])]
        else:
            period_idx = daily_index[daily_index >= reb_date]

        prior_signals = signal_idx[signal_idx < reb_date]
        if len(prior_signals) == 0:
            signal_date = signal_idx[0]
        else:
            signal_date = prior_signals[-1]

        daily.loc[period_idx, ALL_ASSETS] = signal.loc[signal_date, ALL_ASSETS].values

    daily = daily.ffill().bfill()
    daily = normalize_rows(daily)
    return daily

def compute_turnover(daily_weights, rebalance_dates):
    turnover = pd.Series(0.0, index=daily_weights.index)
    prev_w = None

    for dt in rebalance_dates:
        if dt not in daily_weights.index:
            continue

        w = daily_weights.loc[dt, ALL_ASSETS]

        if prev_w is None:
            turnover.loc[dt] = w.drop(labels=[CASH_COL], errors="ignore").abs().sum()
        else:
            turnover.loc[dt] = (w - prev_w).abs().sum() / 2.0

        prev_w = w

    return turnover

def backtest_on_fixed_index(policy_name, signal_weights, etf_returns, evaluation_index):
    evaluation_index = pd.DatetimeIndex(evaluation_index).sort_values()

    returns = etf_returns.copy()
    returns[CASH_COL] = 0.0
    returns = returns.reindex(evaluation_index)

    if returns[ALL_ASSETS].isna().any().any():
        missing_rows = returns[returns[ALL_ASSETS].isna().any(axis=1)]
        raise ValueError(
            f"Return panel has missing rows for {policy_name}. "
            f"Example missing dates: {missing_rows.index[:5].tolist()}"
        )

    signal = signal_weights.copy()
    signal.index = pd.to_datetime(signal.index)
    signal = signal.sort_index()
    signal = signal.reindex(columns=ALL_ASSETS).fillna(0.0)

    signal_aligned = signal.reindex(evaluation_index).ffill().bfill()
    signal_aligned = normalize_rows(signal_aligned)

    rebalance_dates = get_rebalance_dates(evaluation_index, REBALANCE_FREQUENCY)

    daily_weights = expand_rebalance_weights_to_daily(
        signal_weight_df=signal_aligned,
        daily_index=evaluation_index,
        rebalance_dates=rebalance_dates,
    )

    gross_return = (daily_weights[ALL_ASSETS] * returns[ALL_ASSETS]).sum(axis=1)
    turnover = compute_turnover(daily_weights, rebalance_dates)
    transaction_cost = turnover * TRANSACTION_COST_RATE
    net_return = gross_return - transaction_cost

    equity = (1.0 + net_return).cumprod() * INITIAL_CAPITAL
    drawdown = equity / equity.cummax() - 1.0

    out = pd.DataFrame(index=evaluation_index)
    out.index.name = "date"
    out["policy_name"] = policy_name
    out["gross_return"] = gross_return
    out["turnover"] = turnover
    out["transaction_cost"] = transaction_cost
    out["net_return"] = net_return
    out["equity"] = equity
    out["drawdown"] = drawdown
    out["is_rebalance_date"] = out.index.isin(rebalance_dates)

    return out, daily_weights

def safe_index_label(x):
    if hasattr(x, "date"):
        return str(x.date())
    return str(x)

def performance_metrics_from_returns(r):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)

    if n == 0:
        return {}

    equity = (1.0 + r).cumprod()
    drawdown = equity / equity.cummax() - 1.0

    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) else np.nan
    sharpe = float((r.mean() / daily_vol) * np.sqrt(ANNUALIZATION_DAYS)) if daily_vol and daily_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float((r.mean() / downside_vol) * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    max_drawdown = float(drawdown.min())
    calmar = float(annual_return / abs(max_drawdown)) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": safe_index_label(r.index.min()),
        "end_date": safe_index_label(r.index.max()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "hit_rate": float((r > 0).mean()),
        "avg_daily_return": float(r.mean()),
        "daily_volatility": daily_vol,
        "worst_daily_return": float(r.min()),
        "best_daily_return": float(r.max()),
        "final_equity": float(equity.iloc[-1]),
    }

def performance_metrics(return_df):
    return performance_metrics_from_returns(return_df["net_return"])

def constant_weight_signal(index, weights_dict):
    w = pd.DataFrame(0.0, index=pd.DatetimeIndex(index), columns=ALL_ASSETS)
    for k, v in weights_dict.items():
        if k in w.columns:
            w[k] = float(v)
    return normalize_rows(w)

def exposure_diagnostics(weight_df, return_df, label, config=None):
    w = weight_df.reindex(columns=ALL_ASSETS).fillna(0.0)
    w = normalize_rows(w)

    cash_cap = float(config["max_cash_weight"]) if config is not None else np.nan
    cap_tol = 1e-8

    risky = w[ETF_UNIVERSE]
    max_etf_each_day = risky.max(axis=1)

    return {
        "strategy_control": label,
        "avg_cash_weight": float(w[CASH_COL].mean()),
        "median_cash_weight": float(w[CASH_COL].median()),
        "min_cash_weight": float(w[CASH_COL].min()),
        "max_cash_weight_realized": float(w[CASH_COL].max()),
        "pct_days_at_cash_cap": (
            float((w[CASH_COL] >= cash_cap - cap_tol).mean())
            if np.isfinite(cash_cap)
            else np.nan
        ),
        "avg_equity_exposure": float(risky.sum(axis=1).mean()),
        "median_equity_exposure": float(risky.sum(axis=1).median()),
        "avg_0050_weight": float(w["0050"].mean()),
        "avg_006208_weight": float(w["006208"].mean()),
        "avg_00692_weight": float(w["00692"].mean()),
        "avg_00881_weight": float(w["00881"].mean()),
        "avg_max_etf_weight": float(max_etf_each_day.mean()),
        "median_max_etf_weight": float(max_etf_each_day.median()),
        "total_turnover": float(return_df["turnover"].sum()) if "turnover" in return_df else np.nan,
        "avg_turnover": float(return_df["turnover"].mean()) if "turnover" in return_df else np.nan,
        "total_transaction_cost": float(return_df["transaction_cost"].sum()) if "transaction_cost" in return_df else np.nan,
        "num_rebalance_days": int(return_df["is_rebalance_date"].sum()) if "is_rebalance_date" in return_df else np.nan,
    }

def probability_control_diagnostics(features_df, label, original_features=None):
    out = {
        "strategy_control": label,
        "mean_entropy": float(features_df["normalized_entropy"].mean()),
        "std_entropy": float(features_df["normalized_entropy"].std(ddof=1)),
        "mean_bearish_probability": float(features_df["p_bearish"].mean()),
        "std_bearish_probability": float(features_df["p_bearish"].std(ddof=1)),
        "mean_bullish_probability": float(features_df["p_bullish"].mean()),
        "mean_expected_class": float(features_df["expected_class"].mean()),
        "mean_probability_margin": float(features_df["probability_margin"].mean()),
        "mean_ordinal_variance": float(features_df["ordinal_variance"].mean()),
    }

    if original_features is not None:
        common = features_df.index.intersection(original_features.index)
        if len(common) > 1:
            out["corr_entropy_with_original"] = float(
                features_df.loc[common, "normalized_entropy"].corr(
                    original_features.loc[common, "normalized_entropy"]
                )
            )
            out["corr_bearish_with_original"] = float(
                features_df.loc[common, "p_bearish"].corr(
                    original_features.loc[common, "p_bearish"]
                )
            )
        else:
            out["corr_entropy_with_original"] = np.nan
            out["corr_bearish_with_original"] = np.nan
    else:
        out["corr_entropy_with_original"] = np.nan
        out["corr_bearish_with_original"] = np.nan

    return out

# ============================================================
# 6. Bootstrap inference
# ============================================================

def circular_block_indices(n, block_length, rng):
    idx = []
    while len(idx) < n:
        start = int(rng.integers(0, n))
        block = [(start + j) % n for j in range(block_length)]
        idx.extend(block)
    return np.asarray(idx[:n], dtype=int)

def paired_difference_metrics(strategy_returns, comparator_returns):
    s = pd.Series(strategy_returns).dropna().astype(float)
    c = pd.Series(comparator_returns).dropna().astype(float)

    common = s.index.intersection(c.index).sort_values()

    if len(common) > 0:
        s = s.loc[common]
        c = c.loc[common]
    else:
        if len(s) != len(c):
            raise ValueError("No common index and unequal return lengths.")
        s = s.reset_index(drop=True)
        c = c.reset_index(drop=True)

    ms = performance_metrics_from_returns(s)
    mc = performance_metrics_from_returns(c)

    return {
        "n_days": int(len(s)),
        "diff_total_return": ms["total_return"] - mc["total_return"],
        "diff_annual_return": ms["annual_return"] - mc["annual_return"],
        "diff_annual_volatility": ms["annual_volatility"] - mc["annual_volatility"],
        "diff_sharpe": ms["sharpe_ratio"] - mc["sharpe_ratio"],
        "diff_sortino": ms["sortino_ratio"] - mc["sortino_ratio"],
        "drawdown_improvement": ms["max_drawdown"] - mc["max_drawdown"],
        "diff_calmar": ms["calmar_ratio"] - mc["calmar_ratio"],
        "strategy_total_return": ms["total_return"],
        "comparator_total_return": mc["total_return"],
        "strategy_sharpe": ms["sharpe_ratio"],
        "comparator_sharpe": mc["sharpe_ratio"],
        "strategy_sortino": ms["sortino_ratio"],
        "comparator_sortino": mc["sortino_ratio"],
        "strategy_max_drawdown": ms["max_drawdown"],
        "comparator_max_drawdown": mc["max_drawdown"],
    }

def paired_circular_block_bootstrap(strategy_returns, comparator_returns, n_rep=5000, block_length=20, seed=42):
    s = pd.Series(strategy_returns).dropna().astype(float)
    c = pd.Series(comparator_returns).dropna().astype(float)

    common = s.index.intersection(c.index).sort_values()
    if len(common) > 0:
        s = s.loc[common]
        c = c.loc[common]
    else:
        if len(s) != len(c):
            raise ValueError("No common index and unequal return lengths.")
        s = s.reset_index(drop=True)
        c = c.reset_index(drop=True)

    n = len(s)
    if n <= block_length:
        raise ValueError(f"Too few observations for block bootstrap: n={n}, block_length={block_length}")

    observed = paired_difference_metrics(s, c)

    rng = np.random.default_rng(seed)

    metrics = [
        "diff_total_return",
        "diff_sharpe",
        "diff_sortino",
        "drawdown_improvement",
    ]

    dist = {m: [] for m in metrics}
    s_values = s.values
    c_values = c.values

    for _ in range(n_rep):
        idx = circular_block_indices(n=n, block_length=block_length, rng=rng)

        # Integer-indexed bootstrap samples are intentional.
        bs = pd.Series(s_values[idx]).reset_index(drop=True)
        bc = pd.Series(c_values[idx]).reset_index(drop=True)

        bdiff = paired_difference_metrics(bs, bc)

        for m in metrics:
            dist[m].append(bdiff[m])

    rows = []

    for m in metrics:
        arr = np.asarray(dist[m], dtype=float)
        arr = arr[np.isfinite(arr)]

        ci_low, ci_high = np.percentile(arr, [2.5, 97.5])
        obs = float(observed[m])

        if ci_low > 0:
            result = "Significant positive"
        elif ci_high < 0:
            result = "Significant negative"
        else:
            result = "Not significant"

        rows.append({
            "metric": m,
            "observed_difference": obs,
            "ci95_lower": float(ci_low),
            "ci95_upper": float(ci_high),
            "result": result,
            "bootstrap_replications": int(n_rep),
            "block_length": int(block_length),
            "ci_type": "percentile",
        })

    return pd.DataFrame(rows)

# ============================================================
# 7. Load data and probability inputs
# ============================================================

print("\n" + "=" * 80)
print("Loading ETF returns and Notebook 07 probability inputs")
print("=" * 80)

etf_returns, etf_return_path = load_etf_return_panel()

print("ETF return panel:", etf_return_path)
print("ETF return shape:", etf_returns.shape)
print("ETF date range:", etf_returns.index.min().date(), "to", etf_returns.index.max().date())

input_index_df = pd.read_csv(NOTEBOOK08_INPUT_INDEX)

p20_path = None
p60_path = None

for _, row in input_index_df.iterrows():
    target_col = row["target_col"]
    path = Path(row["probability_path_parquet"])
    if not path.exists():
        path = Path(row["probability_path_csv"])

    if "20d" in target_col:
        p20_path = path
    elif "60d" in target_col:
        p60_path = path

if p20_path is None or p60_path is None:
    raise ValueError("Could not locate both 20d and 60d probability files.")

p20_raw = read_table_auto(p20_path)
p60_raw = read_table_auto(p60_path)

for name, prob_df in [("p20_raw", p20_raw), ("p60_raw", p60_raw)]:
    missing = [c for c in proba_cols() if c not in prob_df.columns]
    if missing:
        raise ValueError(f"{name} missing probability columns: {missing}")
    if "fold_id" not in prob_df.columns:
        raise ValueError(f"{name} must include fold_id.")
    if "split" not in prob_df.columns:
        raise ValueError(f"{name} must include split.")

p20_test_latest = latest_fold_deduplicate(p20_raw, split_filter=["test"])
p60_test_latest = latest_fold_deduplicate(p60_raw, split_filter=["test"])

aligned_test_dates = (
    p20_test_latest.index
    .intersection(p60_test_latest.index)
    .intersection(etf_returns.index)
    .sort_values()
)

aligned_test_dates = aligned_test_dates[
    (aligned_test_dates >= pd.Timestamp(STRICT_START))
    & (aligned_test_dates <= pd.Timestamp(STRICT_END))
]

if len(aligned_test_dates) == 0:
    raise ValueError("No aligned strict-test dates found.")

print("Aligned strict-test dates:", len(aligned_test_dates))
print("Date range:", aligned_test_dates.min().date(), "to", aligned_test_dates.max().date())
print("20d probability file:", p20_path)
print("60d probability file:", p60_path)

# Constant-average uncertainty parameters.
p20_oos_latest = latest_fold_deduplicate(p20_raw, split_filter=["validation", "test"])
p60_oos_latest = latest_fold_deduplicate(p60_raw, split_filter=["validation", "test"])

pre_strict_dates = (
    p20_oos_latest.index
    .intersection(p60_oos_latest.index)
    .intersection(etf_returns.index)
    .sort_values()
)
pre_strict_dates = pre_strict_dates[pre_strict_dates < pd.Timestamp(STRICT_START)]

tmp_cfg_for_const = copy.deepcopy(BASE_UAMV_B_CONFIG)

if len(pre_strict_dates) >= 20:
    const_features = probability_features_from_blend(
        p20_oos_latest.loc[pre_strict_dates],
        p60_oos_latest.loc[pre_strict_dates],
        tmp_cfg_for_const,
    )
    constant_average_source = "pre_strict_oos_validation_test_dates"
else:
    const_features = probability_features_from_blend(
        p20_test_latest.loc[aligned_test_dates],
        p60_test_latest.loc[aligned_test_dates],
        tmp_cfg_for_const,
    )
    constant_average_source = "aligned_strict_test_dates_diagnostic_fallback"

CONSTANT_AVERAGE_U = float(const_features["normalized_entropy"].mean())
CONSTANT_AVERAGE_B = float(const_features["p_bearish"].mean())

print("Constant-average U source:", constant_average_source)
print("Constant-average U:", CONSTANT_AVERAGE_U)
print("Constant-average B:", CONSTANT_AVERAGE_B)

# ============================================================
# 8. Define mechanism-control variants
# ============================================================

def make_config(name, label, changes):
    cfg = copy.deepcopy(BASE_UAMV_B_CONFIG)
    cfg["strategy_name"] = name
    cfg["control_label"] = label
    cfg["control_changes"] = changes
    for k, v in changes.items():
        cfg[k] = v
    return cfg

control_configs = []

control_configs.append(
    make_config(
        "MC_ORIGINAL_AURORA10_UAMV_B",
        "AURORA10-UAMV-B original",
        {},
    )
)

control_configs.append(
    make_config(
        "MC_20D_ONLY_DYNAMIC",
        "20d-only AURORA",
        {
            "alpha_20d": 1.00,
            "alpha_60d": 0.00,
        },
    )
)

control_configs.append(
    make_config(
        "MC_60D_ONLY_DYNAMIC",
        "60d-only AURORA",
        {
            "alpha_20d": 0.00,
            "alpha_60d": 1.00,
        },
    )
)

control_configs.append(
    make_config(
        "MC_EQUAL_BLEND_DYNAMIC",
        "Equal-blend AURORA",
        {
            "alpha_20d": 0.50,
            "alpha_60d": 0.50,
        },
    )
)

control_configs.append(
    make_config(
        "MC_CONSTANT_AVERAGE_UNCERTAINTY",
        "Constant-average uncertainty AURORA",
        {
            "risk_aversion_mode": "constant_average",
            "constant_average_U": CONSTANT_AVERAGE_U,
            "constant_average_B": CONSTANT_AVERAGE_B,
        },
    )
)

for seed in SHUFFLE_SEEDS:
    control_configs.append(
        make_config(
            f"MC_SHUFFLED_PROBABILITIES_SEED_{seed}",
            f"Shuffled-probability AURORA seed {seed}",
            {
                "shuffle_probability_seed": seed,
            },
        )
    )

print("\nMechanism-control variants:")
for cfg in control_configs:
    print(cfg["strategy_name"], "|", cfg["control_label"], "|", cfg["control_changes"])

# ============================================================
# 9. Run controls
# ============================================================

print("\n" + "=" * 80)
print("Running mechanism-control variants")
print("=" * 80)

metric_rows = []
exposure_rows = []
probability_diag_rows = []

return_frames = []
weight_frames = []
optimizer_diag_frames = []
probability_feature_frames = []

control_return_series = {}

# Static exposure-matched baseline.
static_label = "Static 44.35% equity / 55.65% cash"
static_policy_name = "MC_STATIC_4435_EQUITY_5565_CASH"

static_signal = constant_weight_signal(aligned_test_dates, STATIC_EXPOSURE_MATCHED_WEIGHTS)

static_return_df, static_weight_df = backtest_on_fixed_index(
    policy_name=static_policy_name,
    signal_weights=static_signal,
    etf_returns=etf_returns,
    evaluation_index=aligned_test_dates,
)

static_metrics = performance_metrics(static_return_df)
static_metrics.update({
    "run_id": RUN_ID,
    "strategy_control": static_label,
    "policy_name": static_policy_name,
    "control_type": "static_exposure_matched_cash_baseline",
    "alpha_20d": np.nan,
    "alpha_60d": np.nan,
    "risk_aversion_mode": "none_static",
})

metric_rows.append(static_metrics)
exposure_rows.append(exposure_diagnostics(static_weight_df, static_return_df, static_label, config=None))

static_return_out = static_return_df.copy()
static_return_out["strategy_control"] = static_label
static_return_out["policy_name"] = static_policy_name
return_frames.append(static_return_out)

static_weight_out = static_weight_df.copy()
static_weight_out.insert(0, "strategy_control", static_label)
static_weight_out.insert(0, "policy_name", static_policy_name)
weight_frames.append(static_weight_out)

control_return_series[static_label] = static_return_df["net_return"].copy()

static_return_df.to_parquet(RETURN_DIR / f"returns_{safe_name(static_policy_name)}.parquet")
static_return_df.to_csv(RETURN_DIR / f"returns_{safe_name(static_policy_name)}.csv")
static_weight_df.to_parquet(WEIGHT_DIR / f"weights_{safe_name(static_policy_name)}.parquet")
static_weight_df.to_csv(WEIGHT_DIR / f"weights_{safe_name(static_policy_name)}.csv")

print("Completed:", static_label, static_metrics)

for cfg in control_configs:
    policy_name = f"AURORA16_{cfg['strategy_name']}"
    label = cfg["control_label"]
    print("\nRunning:", policy_name, "|", label)

    p20_use = p20_test_latest
    p60_use = p60_test_latest

    if "shuffle_probability_seed" in cfg:
        seed = int(cfg["shuffle_probability_seed"])
        p20_use, p60_use = shuffled_probability_inputs(
            p20=p20_test_latest,
            p60=p60_test_latest,
            evaluation_dates=aligned_test_dates,
            seed=seed,
        )

    signal_w, opt_diag_df, prob_features_df = build_uamv_signal_weights(
        p20=p20_use,
        p60=p60_use,
        etf_returns=etf_returns,
        config=cfg,
        evaluation_dates=aligned_test_dates,
    )

    ret_df, daily_w = backtest_on_fixed_index(
        policy_name=policy_name,
        signal_weights=signal_w,
        etf_returns=etf_returns,
        evaluation_index=aligned_test_dates,
    )

    metrics = performance_metrics(ret_df)
    metrics.update({
        "run_id": RUN_ID,
        "strategy_control": label,
        "policy_name": policy_name,
        "control_type": (
            "shuffled_probability_control"
            if "shuffle_probability_seed" in cfg
            else "aurora_mechanism_control"
        ),
        "alpha_20d": cfg["alpha_20d"],
        "alpha_60d": cfg["alpha_60d"],
        "risk_aversion_mode": cfg.get("risk_aversion_mode", "dynamic"),
        "constant_average_U": cfg.get("constant_average_U", np.nan),
        "constant_average_B": cfg.get("constant_average_B", np.nan),
        "shuffle_probability_seed": cfg.get("shuffle_probability_seed", np.nan),
        "base_risk_aversion": cfg["base_risk_aversion"],
        "uncertainty_risk_multiplier": cfg["uncertainty_risk_multiplier"],
        "bearish_risk_multiplier": cfg["bearish_risk_multiplier"],
        "max_cash_weight": cfg["max_cash_weight"],
    })

    metric_rows.append(metrics)
    exposure_rows.append(exposure_diagnostics(daily_w, ret_df, label, cfg))

    ret_out = ret_df.copy()
    ret_out["strategy_control"] = label
    ret_out["policy_name"] = policy_name
    return_frames.append(ret_out)

    w_out = daily_w.copy()
    w_out.insert(0, "strategy_control", label)
    w_out.insert(0, "policy_name", policy_name)
    weight_frames.append(w_out)

    opt_diag_out = opt_diag_df.copy()
    opt_diag_out["strategy_control"] = label
    opt_diag_out["policy_name"] = policy_name
    optimizer_diag_frames.append(opt_diag_out)

    prob_feat_out = prob_features_df.copy()
    prob_feat_out["strategy_control"] = label
    prob_feat_out["policy_name"] = policy_name
    probability_feature_frames.append(prob_feat_out)

    control_return_series[label] = ret_df["net_return"].copy()

    ret_df.to_parquet(RETURN_DIR / f"returns_{safe_name(policy_name)}.parquet")
    ret_df.to_csv(RETURN_DIR / f"returns_{safe_name(policy_name)}.csv")

    daily_w.to_parquet(WEIGHT_DIR / f"weights_{safe_name(policy_name)}.parquet")
    daily_w.to_csv(WEIGHT_DIR / f"weights_{safe_name(policy_name)}.csv")

    opt_diag_df.to_parquet(DIAG_DIR / f"optimizer_diagnostics_{safe_name(policy_name)}.parquet")
    opt_diag_df.to_csv(DIAG_DIR / f"optimizer_diagnostics_{safe_name(policy_name)}.csv")

    print(
        f"  total_return={metrics['total_return']:.4f}, "
        f"sharpe={metrics['sharpe_ratio']:.4f}, "
        f"sortino={metrics['sortino_ratio']:.4f}, "
        f"maxDD={metrics['max_drawdown']:.4f}, "
        f"avg_cash={daily_w[CASH_COL].mean():.4f}"
    )

# Shuffled mean return series.
shuffled_labels = [f"Shuffled-probability AURORA seed {seed}" for seed in SHUFFLE_SEEDS]
shuffled_existing = [lbl for lbl in shuffled_labels if lbl in control_return_series]

if shuffled_existing:
    shuffled_matrix = pd.DataFrame({lbl: control_return_series[lbl] for lbl in shuffled_existing}).sort_index()
    shuffled_mean_returns = shuffled_matrix.mean(axis=1)
    control_return_series["Shuffled-probability AURORA mean"] = shuffled_mean_returns

    mean_metrics = performance_metrics_from_returns(shuffled_mean_returns)
    mean_metrics.update({
        "run_id": RUN_ID,
        "strategy_control": "Shuffled-probability AURORA mean",
        "policy_name": "MC_SHUFFLED_PROBABILITIES_MEAN_SERIES",
        "control_type": "shuffled_probability_control_mean_series",
        "alpha_20d": 0.30,
        "alpha_60d": 0.70,
        "risk_aversion_mode": "dynamic",
        "constant_average_U": np.nan,
        "constant_average_B": np.nan,
        "shuffle_probability_seed": "mean_of_seeds",
        "base_risk_aversion": BASE_UAMV_B_CONFIG["base_risk_aversion"],
        "uncertainty_risk_multiplier": BASE_UAMV_B_CONFIG["uncertainty_risk_multiplier"],
        "bearish_risk_multiplier": BASE_UAMV_B_CONFIG["bearish_risk_multiplier"],
        "max_cash_weight": BASE_UAMV_B_CONFIG["max_cash_weight"],
    })
    metric_rows.append(mean_metrics)

    shuffled_mean_df = pd.DataFrame(index=shuffled_mean_returns.index)
    shuffled_mean_df.index.name = "date"
    shuffled_mean_df["policy_name"] = "MC_SHUFFLED_PROBABILITIES_MEAN_SERIES"
    shuffled_mean_df["net_return"] = shuffled_mean_returns
    shuffled_mean_df["gross_return"] = np.nan
    shuffled_mean_df["turnover"] = np.nan
    shuffled_mean_df["transaction_cost"] = np.nan
    shuffled_mean_df["equity"] = (1.0 + shuffled_mean_returns).cumprod()
    shuffled_mean_df["drawdown"] = shuffled_mean_df["equity"] / shuffled_mean_df["equity"].cummax() - 1.0
    shuffled_mean_df["is_rebalance_date"] = False
    shuffled_mean_df["strategy_control"] = "Shuffled-probability AURORA mean"
    return_frames.append(shuffled_mean_df)

# ============================================================
# 10. Probability diagnostics
# ============================================================

original_features = None

for fdf in probability_feature_frames:
    if "strategy_control" in fdf.columns:
        if fdf["strategy_control"].iloc[0] == "AURORA10-UAMV-B original":
            original_features = fdf.drop(columns=["strategy_control", "policy_name"], errors="ignore")
            break

for fdf in probability_feature_frames:
    label = fdf["strategy_control"].iloc[0]
    clean_f = fdf.drop(columns=["strategy_control", "policy_name"], errors="ignore")
    probability_diag_rows.append(
        probability_control_diagnostics(
            clean_f,
            label=label,
            original_features=original_features,
        )
    )

# ============================================================
# 11. Bootstrap comparisons
# ============================================================

print("\n" + "=" * 80)
print("Running paired circular block-bootstrap mechanism comparisons")
print("=" * 80)

bootstrap_frames = []

primary_label = "AURORA10-UAMV-B original"

bootstrap_comparators = [
    "Static 44.35% equity / 55.65% cash",
    "Constant-average uncertainty AURORA",
    "20d-only AURORA",
    "60d-only AURORA",
    "Equal-blend AURORA",
    "Shuffled-probability AURORA mean",
]

for comp_label in bootstrap_comparators:
    if primary_label not in control_return_series or comp_label not in control_return_series:
        print("Skipping bootstrap comparison; missing:", primary_label, "or", comp_label)
        continue

    print("Bootstrap:", primary_label, "vs", comp_label)

    boot_df = paired_circular_block_bootstrap(
        strategy_returns=control_return_series[primary_label],
        comparator_returns=control_return_series[comp_label],
        n_rep=BOOTSTRAP_REPLICATIONS,
        block_length=BOOTSTRAP_BLOCK_LENGTH,
        seed=BOOTSTRAP_RANDOM_SEED + len(bootstrap_frames),
    )

    boot_df.insert(0, "comparison", f"{primary_label} vs {comp_label}")
    boot_df.insert(1, "strategy_control", primary_label)
    boot_df.insert(2, "comparator_control", comp_label)

    bootstrap_frames.append(boot_df)

if bootstrap_frames:
    bootstrap_df = pd.concat(bootstrap_frames, ignore_index=True)
else:
    bootstrap_df = pd.DataFrame()

# ============================================================
# 12. Export tables and artifacts
# ============================================================

print("\n" + "=" * 80)
print("Exporting Notebook 16 outputs")
print("=" * 80)

metrics_df = pd.DataFrame(metric_rows)

order = [
    "AURORA10-UAMV-B original",
    "Static 44.35% equity / 55.65% cash",
    "Constant-average uncertainty AURORA",
    "20d-only AURORA",
    "60d-only AURORA",
    "Equal-blend AURORA",
    "Shuffled-probability AURORA mean",
] + shuffled_existing

metrics_df["display_order"] = metrics_df["strategy_control"].apply(
    lambda x: order.index(x) if x in order else 999
)
metrics_df = metrics_df.sort_values(["display_order", "strategy_control"]).reset_index(drop=True)

exposure_df = pd.DataFrame(exposure_rows)
exposure_df["display_order"] = exposure_df["strategy_control"].apply(
    lambda x: order.index(x) if x in order else 999
)
exposure_df = exposure_df.sort_values(["display_order", "strategy_control"]).reset_index(drop=True)

prob_diag_df = pd.DataFrame(probability_diag_rows)
prob_diag_df["display_order"] = prob_diag_df["strategy_control"].apply(
    lambda x: order.index(x) if x in order else 999
)
prob_diag_df = prob_diag_df.sort_values(["display_order", "strategy_control"]).reset_index(drop=True)

all_returns_df = pd.concat(return_frames, axis=0).sort_index()
all_weights_df = pd.concat(weight_frames, axis=0).sort_index() if weight_frames else pd.DataFrame()
all_optimizer_diag_df = pd.concat(optimizer_diag_frames, axis=0).sort_index() if optimizer_diag_frames else pd.DataFrame()
all_probability_features_df = pd.concat(probability_feature_frames, axis=0).sort_index() if probability_feature_frames else pd.DataFrame()

s27_cols = [
    "strategy_control",
    "policy_name",
    "control_type",
    "n_days",
    "start_date",
    "end_date",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "hit_rate",
    "avg_daily_return",
    "daily_volatility",
    "worst_daily_return",
    "best_daily_return",
    "final_equity",
    "alpha_20d",
    "alpha_60d",
    "risk_aversion_mode",
    "constant_average_U",
    "constant_average_B",
    "shuffle_probability_seed",
]

s27 = metrics_df[[c for c in s27_cols if c in metrics_df.columns]].copy()
s28 = bootstrap_df.copy()

s29_cols = [
    "strategy_control",
    "mean_entropy",
    "std_entropy",
    "mean_bearish_probability",
    "std_bearish_probability",
    "mean_bullish_probability",
    "mean_expected_class",
    "mean_probability_margin",
    "mean_ordinal_variance",
    "corr_entropy_with_original",
    "corr_bearish_with_original",
]

s29 = prob_diag_df[[c for c in s29_cols if c in prob_diag_df.columns]].copy()

s30_cols = [
    "strategy_control",
    "avg_cash_weight",
    "median_cash_weight",
    "min_cash_weight",
    "max_cash_weight_realized",
    "pct_days_at_cash_cap",
    "avg_equity_exposure",
    "median_equity_exposure",
    "avg_0050_weight",
    "avg_006208_weight",
    "avg_00692_weight",
    "avg_00881_weight",
    "avg_max_etf_weight",
    "median_max_etf_weight",
    "total_turnover",
    "avg_turnover",
    "total_transaction_cost",
    "num_rebalance_days",
]

s30 = exposure_df[[c for c in s30_cols if c in exposure_df.columns]].copy()

outputs = {
    "table_S27_mechanism_control_performance.csv": s27,
    "table_S28_mechanism_control_bootstrap.csv": s28,
    "table_S29_probability_control_diagnostics.csv": s29,
    "table_S30_exposure_control_diagnostics.csv": s30,
}

for fname, df_out in outputs.items():
    local_path = TABLE_RUN_DIR / fname
    global_path = TABLE_DIR / f"{Path(fname).stem}_{RUN_ID}.csv"
    df_out.to_csv(local_path, index=False)
    df_out.to_csv(global_path, index=False)
    print("Saved:", local_path)
    print("Saved:", global_path)

def save_rounded(df, local_name, global_name):
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = out[c].round(6)
    out.to_csv(TABLE_RUN_DIR / local_name, index=False)
    out.to_csv(TABLE_DIR / global_name, index=False)
    return out

s27_round = save_rounded(
    s27,
    "table_S27_mechanism_control_performance_rounded.csv",
    f"table_S27_mechanism_control_performance_rounded_{RUN_ID}.csv",
)

s28_round = save_rounded(
    s28,
    "table_S28_mechanism_control_bootstrap_rounded.csv",
    f"table_S28_mechanism_control_bootstrap_rounded_{RUN_ID}.csv",
)

s29_round = save_rounded(
    s29,
    "table_S29_probability_control_diagnostics_rounded.csv",
    f"table_S29_probability_control_diagnostics_rounded_{RUN_ID}.csv",
)

s30_round = save_rounded(
    s30,
    "table_S30_exposure_control_diagnostics_rounded.csv",
    f"table_S30_exposure_control_diagnostics_rounded_{RUN_ID}.csv",
)

all_returns_df.to_parquet(RETURN_DIR / "all_mechanism_control_returns.parquet")
all_returns_df.to_csv(RETURN_DIR / "all_mechanism_control_returns.csv")

all_weights_df.to_parquet(WEIGHT_DIR / "all_mechanism_control_weights.parquet")
all_weights_df.to_csv(WEIGHT_DIR / "all_mechanism_control_weights.csv")

if not all_optimizer_diag_df.empty:
    all_optimizer_diag_df.to_parquet(DIAG_DIR / "all_mechanism_control_optimizer_diagnostics.parquet")
    all_optimizer_diag_df.to_csv(DIAG_DIR / "all_mechanism_control_optimizer_diagnostics.csv")

if not all_probability_features_df.empty:
    all_probability_features_df.to_parquet(DIAG_DIR / "all_mechanism_control_probability_features.parquet")
    all_probability_features_df.to_csv(DIAG_DIR / "all_mechanism_control_probability_features.csv")

# ============================================================
# 13. Diagnostic plots
# ============================================================

try:
    plot_df = s27.copy()
    central_controls = order[:7]
    plot_df = plot_df[plot_df["strategy_control"].isin(central_controls)].copy()

    plt.figure(figsize=(10, 5))
    plt.barh(plot_df["strategy_control"], plot_df["sharpe_ratio"])
    plt.xlabel("Sharpe ratio")
    plt.title("Mechanism-control Sharpe comparison")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    sharpe_plot = PLOT_DIR / "mechanism_control_sharpe_bar.png"
    plt.savefig(sharpe_plot, dpi=300)
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.barh(plot_df["strategy_control"], plot_df["max_drawdown"])
    plt.xlabel("Maximum drawdown")
    plt.title("Mechanism-control maximum drawdown comparison")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    dd_plot = PLOT_DIR / "mechanism_control_max_drawdown_bar.png"
    plt.savefig(dd_plot, dpi=300)
    plt.close()

    return_matrix = pd.DataFrame({
        label: series
        for label, series in control_return_series.items()
        if label in central_controls
    }).sort_index()

    wealth = (1.0 + return_matrix).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0

    plt.figure(figsize=(11, 6))
    for col in wealth.columns:
        lw = 2.8 if col == "AURORA10-UAMV-B original" else 1.7
        plt.plot(wealth.index, wealth[col], label=col, linewidth=lw)
    plt.axhline(1.0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
    plt.title("Mechanism-control cumulative wealth")
    plt.xlabel("Date")
    plt.ylabel("Cumulative wealth")
    plt.legend(fontsize=8, ncol=2)
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    wealth_plot = PLOT_DIR / "mechanism_control_cumulative_wealth.png"
    plt.savefig(wealth_plot, dpi=300)
    plt.close()

    plt.figure(figsize=(11, 6))
    for col in drawdown.columns:
        lw = 2.8 if col == "AURORA10-UAMV-B original" else 1.7
        plt.plot(drawdown.index, drawdown[col], label=col, linewidth=lw)
    plt.axhline(0.0, linewidth=0.8, color="black")
    plt.title("Mechanism-control drawdowns")
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.legend(fontsize=8, ncol=2)
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    drawdown_plot = PLOT_DIR / "mechanism_control_drawdowns.png"
    plt.savefig(drawdown_plot, dpi=300)
    plt.close()

    print("Plots saved to:", PLOT_DIR)

except Exception as e:
    print("Plotting failed:", repr(e))

# ============================================================
# 14. Interpretation helper
# ============================================================

def get_metric_row(label):
    rows = s27[s27["strategy_control"] == label]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()

original = get_metric_row("AURORA10-UAMV-B original")
static = get_metric_row("Static 44.35% equity / 55.65% cash")
constant = get_metric_row("Constant-average uncertainty AURORA")
shuf_mean = get_metric_row("Shuffled-probability AURORA mean")

interpretation_rows = []

if original and static:
    interpretation_rows.append({
        "diagnostic_question": "Does AURORA beat static exposure-matched cash?",
        "finding": (
            "Yes on Sharpe/Sortino/drawdown"
            if (
                original["sharpe_ratio"] > static["sharpe_ratio"]
                and original["sortino_ratio"] > static["sortino_ratio"]
                and original["max_drawdown"] > static["max_drawdown"]
            )
            else "Mixed or no"
        ),
        "evidence": (
            f"AURORA Sharpe={original['sharpe_ratio']:.4f}, Sortino={original['sortino_ratio']:.4f}, "
            f"MaxDD={original['max_drawdown']:.4f}; Static Sharpe={static['sharpe_ratio']:.4f}, "
            f"Sortino={static['sortino_ratio']:.4f}, MaxDD={static['max_drawdown']:.4f}."
        ),
    })

if original and constant:
    interpretation_rows.append({
        "diagnostic_question": "Does dynamic uncertainty beat constant-average uncertainty?",
        "finding": (
            "Yes on Sharpe/Sortino/drawdown"
            if (
                original["sharpe_ratio"] > constant["sharpe_ratio"]
                and original["sortino_ratio"] > constant["sortino_ratio"]
                and original["max_drawdown"] > constant["max_drawdown"]
            )
            else "Mixed or no"
        ),
        "evidence": (
            f"AURORA dynamic Sharpe={original['sharpe_ratio']:.4f}, Sortino={original['sortino_ratio']:.4f}, "
            f"MaxDD={original['max_drawdown']:.4f}; Constant-average Sharpe={constant['sharpe_ratio']:.4f}, "
            f"Sortino={constant['sortino_ratio']:.4f}, MaxDD={constant['max_drawdown']:.4f}."
        ),
    })

if original and shuf_mean:
    interpretation_rows.append({
        "diagnostic_question": "Does AURORA beat shuffled-probability timing?",
        "finding": (
            "Yes on Sharpe/Sortino/drawdown"
            if (
                original["sharpe_ratio"] > shuf_mean["sharpe_ratio"]
                and original["sortino_ratio"] > shuf_mean["sortino_ratio"]
                and original["max_drawdown"] > shuf_mean["max_drawdown"]
            )
            else "Mixed or no"
        ),
        "evidence": (
            f"AURORA original Sharpe={original['sharpe_ratio']:.4f}, Sortino={original['sortino_ratio']:.4f}, "
            f"MaxDD={original['max_drawdown']:.4f}; Shuffled mean Sharpe={shuf_mean['sharpe_ratio']:.4f}, "
            f"Sortino={shuf_mean['sortino_ratio']:.4f}, MaxDD={shuf_mean['max_drawdown']:.4f}."
        ),
    })

interpretation_df = pd.DataFrame(interpretation_rows)
interpretation_df.to_csv(TABLE_RUN_DIR / "notebook16_interpretation_helper.csv", index=False)
interpretation_df.to_csv(TABLE_DIR / f"notebook16_interpretation_helper_{RUN_ID}.csv", index=False)

print("\nInterpretation helper:")
print(interpretation_df.to_string(index=False))

# ============================================================
# 15. Validation report and manifest
# ============================================================

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "16_AURORA_mechanism_control_experiments_corrected_single_cell",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Mechanism-control experiments to test whether AURORA10-UAMV-B's downside-risk "
        "behavior is explained by dynamic uncertainty-aware allocation beyond static cash exposure, "
        "constant-average uncertainty risk aversion, simple probability blending, and shuffled probabilities."
    ),
    "input_paths": {
        "etf_return_panel": str(etf_return_path),
        "notebook08_or_allocation_input_index": str(NOTEBOOK08_INPUT_INDEX),
        "probability_20d_path": str(p20_path),
        "probability_60d_path": str(p60_path),
    },
    "strict_test_period": {
        "start": STRICT_START,
        "end": STRICT_END,
        "n_days": int(len(aligned_test_dates)),
        "actual_start": str(aligned_test_dates.min().date()),
        "actual_end": str(aligned_test_dates.max().date()),
    },
    "mechanism_controls": {
        "static_exposure_matched_weights": STATIC_EXPOSURE_MATCHED_WEIGHTS,
        "constant_average_U": CONSTANT_AVERAGE_U,
        "constant_average_B": CONSTANT_AVERAGE_B,
        "constant_average_source": constant_average_source,
        "probability_blend_controls": [
            "20d-only",
            "60d-only",
            "equal-blend",
            "original 30/70 blend",
        ],
        "shuffled_probability_seeds": SHUFFLE_SEEDS,
    },
    "bootstrap": {
        "replications": BOOTSTRAP_REPLICATIONS,
        "block_length": BOOTSTRAP_BLOCK_LENGTH,
        "random_seed": BOOTSTRAP_RANDOM_SEED,
        "ci_type": "percentile",
        "bootstrap_type": "paired circular block bootstrap",
    },
    "base_config": BASE_UAMV_B_CONFIG,
    "has_scipy_optimizer": HAS_SCIPY,
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "returns": str(RETURN_DIR),
        "weights": str(WEIGHT_DIR),
        "diagnostics": str(DIAG_DIR),
        "plots": str(PLOT_DIR),
        "table_S27": str(TABLE_RUN_DIR / "table_S27_mechanism_control_performance.csv"),
        "table_S28": str(TABLE_RUN_DIR / "table_S28_mechanism_control_bootstrap.csv"),
        "table_S29": str(TABLE_RUN_DIR / "table_S29_probability_control_diagnostics.csv"),
        "table_S30": str(TABLE_RUN_DIR / "table_S30_exposure_control_diagnostics.csv"),
    },
    "educational_note": (
        "This notebook performs research backtests only and does not provide personalized financial advice."
    ),
}

validation_report_path = REPORT_RUN_DIR / "NOTEBOOK16_mechanism_control_validation_report.json"
validation_report_global_path = REPORT_DIR / f"NOTEBOOK16_mechanism_control_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "NOTEBOOK16_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"NOTEBOOK16_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 16. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF NOTEBOOK 16 MECHANISM CONTROLS COMPLETE")
print("=" * 80)
print("Run ID:", RUN_ID)
print("Run root:", RUN_ROOT)
print("Table S27:", TABLE_RUN_DIR / "table_S27_mechanism_control_performance.csv")
print("Table S28:", TABLE_RUN_DIR / "table_S28_mechanism_control_bootstrap.csv")
print("Table S29:", TABLE_RUN_DIR / "table_S29_probability_control_diagnostics.csv")
print("Table S30:", TABLE_RUN_DIR / "table_S30_exposure_control_diagnostics.csv")
print("Returns:", RETURN_DIR / "all_mechanism_control_returns.parquet")
print("Weights:", WEIGHT_DIR / "all_mechanism_control_weights.parquet")
print("Validation report:", validation_report_path)
print("Manifest:", manifest_path)
print("=" * 80)

print("\nRounded Table S27 preview:")
print(s27_round.to_string(index=False))

print("\nRounded Table S28 preview:")
if not s28_round.empty:
    print(s28_round.to_string(index=False))
else:
    print("No bootstrap rows.")

print("\nRounded Table S29 preview:")
print(s29_round.to_string(index=False))

print("\nRounded Table S30 preview:")
print(s30_round.to_string(index=False))

Mounted at /content/drive
AURORA-TWETF Notebook 16: Mechanism-control experiments
RUN_ID: 20260714_084419
RUN_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_mechanism_controls/run_20260714_084419
Notebook 07 input index: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv

Loading ETF returns and Notebook 07 probability inputs
ETF return panel: /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
ETF return shape: (1426, 4)
ETF date range: 2021-01-01 to 2026-06-23
Aligned strict-test dates: 319
Date range: 2024-11-27 to 2026-03-25
20d probability file: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/probabilities/selected_probabilities_TAIEX_regime_fixed_20d_E1_validation_weighted_probability_ensemble.parquet
60d probability file: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA